In [ ]:
import pandas as pd
import numpy as np

df_vendas = pd.read_csv("vendas_tech.csv", low_memory=False)
display(df_vendas)
df_gerentes = pd.read_excel("gerentes_lojas.xlsx")
display(df_gerentes)

In [ ]:
# inspecao
display(df_vendas)
#display(df_vendas.head(7))
display(df_vendas.tail(7))


In [ ]:
#display(df_vendas.sample(15))
#display(df_vendas.shape)
display(list(df_vendas.columns))

In [ ]:
#display(df_vendas.shape)
#display(list(df_vendas.columns))
display(df_vendas.info())
display(df_vendas.describe())

In [ ]:
#tratamento de daods
#display(df_vendas["Loja"])
#display(df_vendas[["Loja", "Cliente"]])

#colunas
df_analise = df_vendas.drop(columns=["Data_Base"])

#nulos
#df_analise = df_analise.dropna() # excluir todas as linhas que tem 1 valor vazio
df_analise["Loja"] = df_analise["Loja"].fillna("Online")

#tipos de dados
df_analise["Data"] = pd.to_datetime(df_analise["Data"], format="%Y-%m-%d")

# padronizacao
df_analise["Loja"] = df_analise["Loja"].str.strip()
df_analise["Loja"] = df_analise["Loja"].str.title()

df_gerentes["Loja"] = df_gerentes["Loja"].str.strip()
df_gerentes["Loja"] = df_gerentes["Loja"].str.title()

#encadeamento
# df_analise["Loja"] = df_analise["Loja"].fillna("Online").str.strip().str.title()

#duplicata
df_analise = df_analise.drop_duplicates(subset=["ID_Pedido"])



display(df_analise)
display(df_analise.info())

In [ ]:
# criar novas colunas
# faturamento
df_analise["Faturamento"] = df_analise['Qtd'] * df_analise["Preco_Unitario"]

# forma de venda
df_analise["Forma_de_Vendas"] = np.where(df_analise["Loja"] == "Online", "Online","Presencial")

# regiao
display(df_analise["Loja"].unique())

dic_regioes = {
    'São Paulo': "Sudeste",
    'Belo Horizonte': "Sudeste",
    'Online': "Online",
    'Rio De Janeiro': "Sudeste",
    'Salvador': "Nordeste",
    'Recife': "Nordeste",
    'Curitiba': "Sul",
    'Porto Alegre': "Sul"
}
df_analise["Regiao"] = df_analise["Loja"].map(dic_regioes)

display(df_analise)
display(df_analise.isna().sum())

In [ ]:
# analise -> filtrar

df_analise = df_analise.sort_values(by=["Data", "Faturamento"])
df_analise = df_analise.reset_index(drop=True)

#.loc: id do pedido -> por nome da coluna e por nome da linha
id_pedido = 4

loja = df_analise.loc[df_analise["ID_Pedido"]==4, "Loja"].values[0]
produto = df_analise.loc[df_analise["ID_Pedido"]==4, "Produto"].values[0]
cliente = df_analise.loc[df_analise["ID_Pedido"]==4, "Cliente"].values[0]
print(loja)
print(produto)
print(cliente)

#.iloc -> por posicao
id_pedido = 4
loja = df_analise.iloc[3,2]
produto = df_analise.iloc[3,3]
cliente = df_analise.iloc[3,6]

print(loja , produto, cliente)


#condicional
df_id_pedido_4 = df_analise[df_analise["ID_Pedido"]==4]
display(df_id_pedido_4)

#exportar pedacos da base
df_vendas_sp = df_analise[df_analise["Loja"]=="São Paulo"]
df_vendas_sp.to_csv("Vendas_SP.csv", index=False)

#exportar as vendas de 2024
df_vendas_2024 = df_analise[df_analise["Data"]>="2024-01-01"]

# duplas condicoes
condicao1 = df_analise["Produto"]=="Cabo HDMI"
condicao2 = df_analise["Regiao"]=="Sul"
df_vendas_HDMI_SUL= df_analise[condicao1 & condicao2]


display(df_vendas_HDMI_SUL)

In [ ]:
# analises por agrupamentos 
#display(df_analise)

# ranking de faturamentos por loja
analise_lojas = df_analise[["Loja", "Faturamento"]].groupby("Loja").sum()
analise_lojas = analise_lojas.sort_values(by="Faturamento", ascending=False)
analise_lojas = analise_lojas.reset_index()
analise_lojas["Faturamento"] = analise_lojas["Faturamento"].map("R${:,.2F}".format)
display(analise_lojas)


# ranking de produtos que mais venderam no online
df_vendas_online = df_analise[df_analise["Loja"]=="Online"]
analise_produtos_online = df_vendas_online[["Produto", "Qtd"]].groupby("Produto").sum()
analise_produtos_online = analise_produtos_online.sort_values(by="Qtd", ascending=False)
analise_produtos_online = analise_produtos_online.rename(columns={"Qtd": "Vendas Totais"}) # Alterar nome de coluna
display(analise_produtos_online)

# analise de ranking por loja e por produto
# quais produtos vederam mais em cada uma das lojas
analise_produtos_em_lojas = df_analise[["Loja","Produto","Qtd"]].groupby(["Loja", "Produto"]).sum()

# quais lojas mais venderam os produtos
analise_lojas_em_produtos = df_analise[["Loja","Produto","Qtd"]].groupby(["Produto", "Loja"]).sum()

with pd.option_context("display.max_rows", None):
    display(analise_produtos_em_lojas)
    display(analise_lojas_em_produtos)


In [ ]:
#display(df_analise.head())

# quais gerentes bateram a meta em janeiro de 2023
df_meta = df_analise[(df_analise["Data"].dt.year==2023) & (df_analise["Data"].dt.month==1)]

df_meta = df_meta[["Loja", "Faturamento"]].groupby("Loja", as_index=False).sum()

#df_meta = df_meta.merge(df_gerentes, left_on="Loja", right_on="Cidade") se as tabelas tiverem nomes diferentes
df_meta = df_meta.merge(df_gerentes, on="Loja", how="left")
df_meta["Bateu Meta"] =np.where(df_meta["Faturamento"] >= df_meta["Meta_Mensal"], "Sim", "Não")
display(df_meta)

In [ ]:
df_analise["Mes-Ano"] = df_analise["Data"].dt.to_period("M")
df_vendas_mes = df_analise[["Mes-Ano", "Faturamento"]].groupby("Mes-Ano").sum()
df_vendas_mes.plot()
display(df_vendas_mes)
